In [0]:
%pip install -U -qqqq databricks-langchain langgraph-supervisor
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks

supervisor_model = ChatDatabricks(endpoint="gpt-4-1")
small_model = ChatDatabricks(endpoint="gpt-4-1-nano")


In [0]:
import mlflow

mlflow.langchain.autolog()

In [0]:
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
import random

In [0]:
def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name} for ${random.randint(10, 100)}."

hotel_agent = create_react_agent(
    model=small_model,
    tools=[book_hotel],
    prompt=("You are a hotel booking assistant\n\n"
        "INSTRUCTIONS:\n"
        "- After you're done with your tasks, respond to the supervisor directly\n"
        "- Respond ONLY with the results of your work, do NOT include ANY other text."),
    name="hotel_agent"
)

In [0]:
def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport} for ${random.randint(10, 100)}."

flight_agent = create_react_agent(
    model=small_model,
    tools=[book_flight],
    prompt=("You are a flight booking assistant\n\n"
        "INSTRUCTIONS:\n"
        "- After you're done with your tasks, respond to the supervisor directly\n"
        "- Respond ONLY with the results of your work, do NOT include ANY other text."),
    name="flight_agent"
)

In [0]:
def add(a: float, b: float):
    """Add two numbers."""
    return a + b


def multiply(a: float, b: float):
    """Multiply two numbers."""
    return a * b


def divide(a: float, b: float):
    """Divide two numbers."""
    return a / b


math_agent = create_react_agent(
    model=small_model,
    tools=[add, multiply, divide],
    prompt=(
        "You are a math agent.\n\n"
        "INSTRUCTIONS:\n"
        "- Assist ONLY with math-related tasks\n"
        "- After you're done with your tasks, respond to the supervisor directly\n"
        "- Respond ONLY with the results of your work, do NOT include ANY other text."
    ),
    name="math_agent",
)

In [0]:
supervisor = create_supervisor(
    agents=[flight_agent, hotel_agent, math_agent],
    model=supervisor_model,
    prompt=(
        "You are a supervisor managing three agents:\n"
        "- hotel booking agent. Assign hotel-related tasks to this agent\n"
        "- flight booking agent. Assign flight-related tasks to this agent\n"
        "- a math agent. Assign math-related tasks to this assistant\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself."
    )
).compile()

In [0]:
from langchain_core.messages import convert_to_messages


def pretty_print_message(message, indent=False):
    pretty_message = message.pretty_repr(html=True)
    if not indent:
        print(pretty_message)
        return

    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)


def pretty_print_messages(update, last_message=False):
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        # skip parent graph updates in the printouts
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        print(f"Update from subgraph {graph_id}:")
        print("\n")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"Update from node {node_name}:"
        if is_subgraph:
            update_label = "\t" + update_label

        print(update_label)
        print("\n")

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)
        print("\n")

In [0]:
for chunk in supervisor.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "book a flight from BOS to JFK and a stay at McKittrick Hotel"
            }
        ]
    }
):
    pretty_print_messages(chunk, last_message=True)

In [0]:
from langgraph.types import Send
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langgraph.graph import StateGraph, START,  MessagesState
from langgraph.types import Command
from langchain.tools import tool
def create_task_description_handoff_tool(
    *, agent_name: str, description: str | None = None
):
    name = f"transfer_to_{agent_name}"
    description = description or f"Ask {agent_name} for help."

    @tool(name, description=description)
    def handoff_tool(
        # this is populated by the supervisor LLM
        task_description: Annotated[
            str,
            "Description of what the next agent should do, including all of the relevant context.",
        ],
        # these parameters are ignored by the LLM
        state: Annotated[MessagesState, InjectedState],
    ) -> Command:
        task_description_message = {"role": "user", "content": task_description}
        agent_input = {**state, "messages": [task_description_message]}
        return Command(
            goto=[Send(agent_name, agent_input)],
            graph=Command.PARENT,
        )

    return handoff_tool


In [0]:
assign_to_math_agent_with_description = create_task_description_handoff_tool(
    agent_name="math_agent",
    description="Assign task to a math agent.",
)
assign_to_flight_agent_with_description = create_task_description_handoff_tool(
    agent_name="flight_agent",
    description="Assign task to a flight agent.",
)
assign_to_hotel_agent_with_description = create_task_description_handoff_tool(
    agent_name="hotel_agent",
    description="Assign task to a hotel agent.",
)

In [0]:
supervisor_agent_with_description = create_react_agent(
    model=supervisor_model,
    tools=[
        assign_to_math_agent_with_description,
        assign_to_flight_agent_with_description,
        assign_to_hotel_agent_with_description,
    ],
    prompt=(
        "You are a supervisor managing three agents:\n"
        "- hotel booking agent. Assign hotel-related tasks to this agent\n"
        "- flight booking agent. Assign flight-related tasks to this agent\n"
        "- a math agent. Assign math-related tasks to this assistant\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself. \n"
        "Give the final sumarized response back to the user."
    ),
    name="supervisor",
)

supervisor_with_description = (
    StateGraph(MessagesState)
    .add_node(
        supervisor_agent_with_description, destinations=("hotel_agent", "flight_agent", "math_agent")
    )
    .add_node(flight_agent)
    .add_node(math_agent)
    .add_node(hotel_agent)
    .add_edge(START, "supervisor")
    .add_edge("flight_agent", "supervisor")
    .add_edge("math_agent", "supervisor")
    .add_edge("hotel_agent", "supervisor")
    .compile()
)

In [0]:
for chunk in supervisor_with_description.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "book a flight from BOS to JFK and a stay at McKittrick Hotel. Then give me a total booking amount."
            }
        ]
    },
    subgraphs=True,
):
    pretty_print_messages(chunk, last_message=True)